# Workflow Development and Testing - MILESTONE 5

As a reminder, AdaptEd helps teachers and parents quickly generate personalized and adaptive lesson plans/activities for students with learning differences.

## Part 1: Workflow development

Based on the usage scenarios from Milestone 2, AdaptEd's routers will send requests to three main categories:
1. Creating a full lesson plan for a student
3. Updating or adapting an existing plan based on new data (e.g. the student's performance on benchmark exams)
4. Generating a quick activity or accomodation idea (e.g. to refocus a student with ADHD after they've experienced a trigger)

The LLM nodes ("brains") will be:
1. Router #1: Categorize request
- Here, the LLM will try and identify what the user is trying to do.
2. Router #2: Detect subject area
- Here, the LLM will identify the learning area (e.g. reading, math, etc.)
3. Planner LLMs (one per branch)
- Here, the LLM will take in what they've learned so far about the student (learning style, subject area, challenges, constraints, etc.) and actually generate the lesson plan
4. Checker/Reflection LLM: reflect on the student's progress and identify strengths/weaknesses
- Here, the LLM will reflect on performance to suggest a revised plan, along with critiques/notes/improvements

Therefore, the overall workflow will look something like this:
First, the user describes the student and goal in natural language. Then, Router LLM #1 extracts information for the student profile, goals, etc. and classifies the request as either one to create a new plan or update the current plan. This request then gets passed to Router LLM #2, which labels the subject area and calls the appropriate Planner LLM for that specific subject area if the request was to create a new plan. If the request was to update the current plan, the message gets passed to the Checker/Reflection LLM. 

### Mermaid diagram:

```mermaid
flowchart TD
    %% Direction: Top to Bottom

    %% User input
    U[User describes student and goal]

    %% Router #1: categorize request
    R1{Router #1<br/>Categorize request}

    %% Router #2: detect subject area
    R2{Router #2<br/>Detect subject area}

    %% Planner LLMs
    P_read[Reading Planner LLM]
    P_math[Math Planner LLM]
    P_write[Writing Planner LLM]

    %% Checker / Reflection LLM
    CH[Checker / Reflection LLM<br/>Reflect on progress<br/>Suggest revisions]

    %% Output
    OUT[(Structured lesson plan<br/>or updated plan)]

    %% Edges
    U --> R1

    R1 -->|Create new plan| R2
    R1 -->|Update current plan| CH

    R2 -->|Reading| P_read
    R2 -->|Math| P_math
    R2 -->|Writing| P_write

    P_read --> OUT
    P_math --> OUT
    P_write --> OUT

    CH --> OUT

## Part 2: Implementation and testing of workflow

In [3]:
from typing_extensions import TypedDict, Literal
from typing import List

from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

from dotenv import load_dotenv
_ = load_dotenv()

In [4]:
llm = ChatAnthropic(model="claude-3-7-sonnet-latest")

In [5]:
from typing_extensions import TypedDict

class State(TypedDict):
    request: str
    route_type: str
    subject: str
    plan: str
    revised_plan: str

In [6]:
def router1(state: State):
    prompt = f"""
You are Router #1.

Decide whether this user request is asking for:
1. a NEW lesson plan
2. an UPDATE to an existing plan

Respond with only one word: new or update.

Request:
{state["request"]}
"""
    result = llm.invoke(prompt)
    return {"route_type": result.content.strip()}


def router2(state: State):
    prompt = f"""
You are Router #2.

Identify the subject area of this request.
Choose from:
- reading
- math
- writing
- exec

Respond with only the subject name.

Request:
{state["request"]}
"""
    result = llm.invoke(prompt)
    return {"subject": result.content.strip()}

In [7]:
def planner(state: State):
    prompt = f"""
Create a simple lesson plan for this request.
Subject: {state['subject']}

Write:
- objective
- materials
- 3–4 clear steps
- one accommodation
"""
    result = llm.invoke(prompt)
    return {"plan": result.content}

In [8]:
def checker(state: State):
    prompt = f"""
You are the Checker LLM.

The user is asking to update or revise a plan.
Give:
- what needs improvement
- how to adjust the plan
- a short revised summary
"""
    result = llm.invoke(prompt)
    return {"revised_plan": result.content}

In [9]:
def route_after_router1(state: State):
    if state["route_type"] == "new":
        return "go_to_router2"
    else:
        return "go_to_checker"

def route_after_router2(state: State):
    return "go_to_planner"

In [10]:
workflow = StateGraph(State)

workflow.add_node("router1", router1)
workflow.add_node("router2", router2)
workflow.add_node("planner", planner)
workflow.add_node("checker", checker)

# Start -> Router #1
workflow.add_edge(START, "router1")

# Router #1 -> new or update
workflow.add_conditional_edges(
    "router1",
    route_after_router1,
    {
        "go_to_router2": "router2",
        "go_to_checker": "checker",
    }
)

# Router #2 -> Planner
workflow.add_conditional_edges(
    "router2",
    route_after_router2,
    {
        "go_to_planner": "planner"
    }
)

# Planner and Checker both -> END
workflow.add_edge("planner", END)
workflow.add_edge("checker", END)

app = workflow.compile()

In [12]:
scenario = """
I need a 30-minute reading lesson for my student who struggles with ADHD.
"""

initial_state = {
    "request": scenario,
    "route_type": "",
    "subject": "",
    "plan": "",
    "revised_plan": ""
}

result = app.invoke(initial_state)
result

{'request': '\nI need a 30-minute reading lesson for my student who struggles with ADHD.\n',
 'route_type': 'new',
 'subject': 'reading',
 'plan': '# Reading Lesson Plan: Identifying Main Ideas and Supporting Details\n\n## Objective\nBy the end of this lesson, students will be able to identify the main idea and at least three supporting details in a grade-level appropriate text.\n\n## Materials\n- Short reading passage for each student (1-2 paragraphs)\n- Main idea and supporting details graphic organizers\n- Highlighters (two different colors)\n- Whiteboard and markers\n- Optional: projector to display example text\n\n## Steps\n\n### Step 1: Introduction and Modeling (10 minutes)\n- Begin by explaining the concept of main idea (the most important point the author is making) and supporting details (information that backs up or explains the main idea).\n- Display a sample paragraph on the board and think aloud as you identify the main idea and supporting details.\n- Highlight the main i